In [1]:
# environment: /glade/work/dkimpara/conda-envs/xesmf

In [2]:
import xarray as xr
import shutil

import os
from os.path import join
import glob
import numpy as np
import datetime

from joblib import Parallel, delayed
import joblib

from glob import glob

from functools import partial
import dask.array as da

import pandas as pd


In [3]:
top_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset"

channels = [4, 7, 8, 9, 10, 13]

times = np.load("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/intermediate_files/2025_times.npy")

In [4]:
# get example ds
ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025_test/2025-01-01_00Z_C04.nc")

darray = ds.BT_or_R

In [5]:
# data shape
new_shape = (len(channels),
             len(times),
             darray.shape[-2],
             darray.shape[-1],
            )

chunks = (len(channels),
          1,
          darray.shape[-2],
          darray.shape[-1],
          )

single_value_shape = (len(channels),
                      len(times))
single_value_chunks = (len(channels),
                       1)


In [6]:
dummies = da.zeros(new_shape,chunks=chunks)
dummies1 = da.zeros(single_value_shape, chunks=single_value_chunks)

In [7]:
zarr_ds = xr.Dataset({"BT_or_R": (("channel", "t", "latitude", "longitude"), dummies),
                      "yaw_flip_flag": (("channel", "t"), dummies1),
                      "BT_or_R_mean": (("channel", "t"), dummies1),
                     },
                     coords={"channel": channels,
                             "t": times,
                             "latitude": ds.lat.values, #grid.latitude,
                             "longitude": ds.lon.values #grid.longitude,
                            }
                    )

In [8]:
save_path = join(top_dir, "goes_10km_2025.zarr")
zarr_ds.to_zarr(save_path, compute=False, consolidated=False,
          mode="w")


Delayed('_finalize_store-62ff0b99-2ccd-43a5-a266-7089a94a225f')

In [9]:
zarr_ds

<xarray.Dataset> Size: 2TB
Dimensions:        (channel: 6, t: 39342, latitude: 1003, longitude: 923)
Coordinates:
  * channel        (channel) int64 48B 4 7 8 9 10 13
  * t              (t) datetime64[ns] 315kB 2025-04-01T19:35:05.535825920 ......
  * latitude       (latitude) float64 8kB -50.1 -50.0 -49.9 ... 49.9 50.0 50.1
  * longitude      (longitude) float64 7kB -121.1 -121.0 -120.9 ... -29.0 -28.9
Data variables:
    BT_or_R        (channel, t, latitude, longitude) float64 2TB dask.array<chunksize=(6, 1, 1003, 923), meta=np.ndarray>
    yaw_flip_flag  (channel, t) float64 2MB dask.array<chunksize=(6, 1), meta=np.ndarray>
    BT_or_R_mean   (channel, t) float64 2MB dask.array<chunksize=(6, 1), meta=np.ndarray>